In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
cd "/content/drive/MyDrive/Courses/AI/masked_attention/llama_like/excercises"

/content/drive/MyDrive/Courses/AI/masked_attention/llama_like/excercises


In [10]:
import sys
sys.path.append("/content/drive/MyDrive/Courses/AI/masked_attention/llama_like")

In [1]:
"""
Taller: Internals de un LLM estilo LLaMA
=================================================================
Instrucciones:
  - Busca los bloques marcados con TODO y completa el código.
  - Cada sección tiene una celda de verificación al final.
  - No modifiques nada fuera de los bloques TODO.

Secciones:
  1. RMSNorm y SwiGLU desde cero
"""

'\nTaller: Internals de un LLM estilo LLaMA\n=================================================================\nInstrucciones:\n  - Busca los bloques marcados con TODO y completa el código.\n  - Cada sección tiene una celda de verificación al final.\n  - No modifiques nada fuera de los bloques TODO.\n \nSecciones:\n  1. RMSNorm y SwiGLU desde cero\n'

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

# Importamos el modelo de referencia para comparaciones
from src.model import (
    ModelConfig, RMSNorm, SwiGLUFFN, MiniLLaMA,
    precompute_rope_freqs, apply_rope, GroupedQueryAttention
)
from src.data import get_corpus
from src.tokenizer import BPETokenizer

In [12]:
# ===========================================================================
# SECCIÓN 1 — RMSNorm y SwiGLU desde cero
# ===========================================================================
# Teoría:
#   RMSNorm(x) = x / sqrt(mean(x²) + eps) * gamma
#
#   SwiGLU(x)  = SiLU(gate_proj(x)) * up_proj(x)
#   donde SiLU(x) = x * sigmoid(x)
#   luego se proyecta de vuelta con down_proj.
# ===========================================================================

class MyRMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps   = eps
        # TODO 1.1 — Define el parámetro learnable gamma (vector de unos, shape: d_model)
        # self.gamma = ...
        raise NotImplementedError("TODO 1.1")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 1.2 — Implementa RMSNorm
        # Pista: usa x.pow(2).mean(dim=-1, keepdim=True) para calcular el RMS
        # return ...
        raise NotImplementedError("TODO 1.2")


class MySwiGLUFFN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        D, Dff = config.d_model, config.d_ff
        # TODO 1.3 — Define las tres proyecciones lineales sin bias:
        #   gate_proj : D → Dff
        #   up_proj   : D → Dff
        #   down_proj : Dff → D
        raise NotImplementedError("TODO 1.3")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 1.4 — Implementa SwiGLU
        # Pista: F.silu() es la función SiLU
        # return ...
        raise NotImplementedError("TODO 1.4")


# ── Verificación 1 ──────────────────────────────────────────────────────────
def verify_section1():
    print("=" * 55)
    print("VERIFICACIÓN 1 — RMSNorm y SwiGLU")
    print("=" * 55)
    cfg = ModelConfig(vocab_size=256, d_model=32, n_heads=4,
                      n_kv_heads=2, d_ff=64, max_seq_len=16)
    x   = torch.randn(2, 8, 32)

    # RMSNorm
    ref_norm  = RMSNorm(32)
    my_norm   = MyRMSNorm(32)
    # Copiar pesos para comparación justa
    my_norm.gamma = ref_norm.gamma
    out_ref   = ref_norm(x)
    out_mine  = my_norm(x)
    norm_ok   = torch.allclose(out_ref, out_mine, atol=1e-5)
    print(f"  RMSNorm   output match : {norm_ok}")
    print(f"  RMSNorm   norm preservada: {torch.allclose(x.norm(dim=-1), out_mine.norm(dim=-1) / my_norm.gamma.norm(), atol=1e-3)}")

    # SwiGLU
    ref_ffn  = SwiGLUFFN(cfg)
    my_ffn   = MySwiGLUFFN(cfg)
    my_ffn.gate_proj.weight = ref_ffn.gate_proj.weight
    my_ffn.up_proj.weight   = ref_ffn.up_proj.weight
    my_ffn.down_proj.weight = ref_ffn.down_proj.weight
    out_ref  = ref_ffn(x)
    out_mine = my_ffn(x)
    ffn_ok   = torch.allclose(out_ref, out_mine, atol=1e-5)
    print(f"  SwiGLU    output match : {ffn_ok}")
    print(f"  SwiGLU    output shape : {list(out_mine.shape)}  (esperado [2, 8, 32])")
    if norm_ok and ffn_ok:
        print("\n  ✓ Sección 1 correcta")
    else:
        print("\n  ✗ Revisa tu implementación")

In [13]:
verify_section1()

VERIFICACIÓN 1 — RMSNorm y SwiGLU


NotImplementedError: TODO 1.1